# Example Filecount Tracker

### Made by Sam Bailey, partially with code adapted from Kyle Lesinger

This little object can be added to any script that changes the number of files in the drcs_activations_new s3 bucket! He's very friendly, so make sure to treat him well.

In [1]:
# Needed Python imports
import sys
from pathlib import Path

In [2]:
#Local imports
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Your little manager has been added right here!
from aws_s3_utils import (
    s3_count_manager
)

Initalizing your little manager is very easy, all you have to do is call him!

```python
manager = s3_count_manager()
```

He'll take a few seconds to look through the inventory, and gather up all the data, but he won't tell you right away what he found:

In [3]:
manager = s3_count_manager()

If you want to know what he found directly, you can just ask him.

```python
manager.current
```

He'll record which top-level directories (satellites) he found, and then how many of each temporal resolution .tif files were in those directories: 'day' means the file ended in "_day", 'monthly' means it ended in "_monthly", and 'subdaily' means the file ended with a timestamp directly (i.e., ended in "Z").

In [4]:
manager.current

,satellite,temporal_resolution,count
0,ALOS2,day,2
1,ALOS2,monthly,1
2,Blackmarble,subdaily,3
3,Blackmarble,day,266
4,Blackmarble,monthly,27
5,ECOSTRESS,subdaily,90
6,GoogleEarth,monthly,1
7,IMERG,day,2
8,IMERG,monthly,7
9,Landsat,subdaily,3


This is obvously super helpful, thanks manager! You can tell him to check again at any time, too.

```python
manager.reevaluate()
```

This will automatically update his `.current` attribute, and will store his previous inventory under `.old` too. But the most helpful thing? Every time you ask him to reevaluate, he ALSO returns with what was different from the last time!

```python
diff = manager.reevaluate()
```

In [5]:
diff = manager.reevaluate()
diff

,satellite,temporal_resolution,old_count,new_count,difference


But, well, right now he doesn't have any differences to report, now does he?
To test his output, we're going to doctor his records a bit, and say he actually only had 287 daily Landsat files, and 1200 daily Sentinel-2 files.

In [6]:
manager.current.loc[manager.row_index_by_product("Landsat", "day"), "count"] = 287
manager.current.loc[manager.row_index_by_product("Sentinel-2", "day"), "count"] = 1200

manager.current

,satellite,temporal_resolution,count
0,ALOS2,day,2
1,ALOS2,monthly,1
2,Blackmarble,subdaily,3
3,Blackmarble,day,266
4,Blackmarble,monthly,27
5,ECOSTRESS,subdaily,90
6,GoogleEarth,monthly,1
7,IMERG,day,2
8,IMERG,monthly,7
9,Landsat,subdaily,3


Now that we've successfully doctored his records, let's have him reevaluate again and see what the difference is between the doctored records and the REAL current state.

In [7]:
diff = manager.reevaluate()
diff

,satellite,temporal_resolution,old_count,new_count,difference
10,Landsat,day,287.0,347.0,60.0
21,Sentinel-2,day,1200.0,1213.0,13.0


Awesome, thanks manager!

If you want him to print out a report of the current, old, or difference DataFrames, you can always run DataFrame.to_csv(filename):

```python
manager.current.to_csv(filename)
manager.old.to_csv(filename)
manager.reevaluate().to_csv(filename)
```

However, you can also directly call `.to_csv()` on him, and he'll automatically give you the current state as `"drcs_activations_new_current_filecounts.csv"` (you can also pass it a filename if you want a different name). This is probably more convenient for most purposes.

```python
manager.to_csv()
```

In [8]:
manager.to_csv()

And that's all you should need to get your manager up and running your filecount changes. I've included a general flowchart below to gather everthing we've learned.

```python
manager = s3_count_manager() # Initialize the manager

### Code here that adds/removes files from the drcs_activations_new s3 bucket

diff = manager.reevaluate() # Compare your changes against the old state
print(diff) # Print those findings

manager.to_csv("example.csv") # Save the current state out to a csv
```

Keep in mind that you can run `manager.reevaluate()` as many times as you want without having to reinitialize him. Additionally, if you already have a version of a different manager's output saved, you can initialize a manager off of that as well.

```python
manager = s3_count_manager("old_filecount.csv")
```

While not usually helpful, if there's ever a case where you want to compare the current state with an old savestate, you can initialize off an old file and immediately run `.reevaluate()`. Have fun with your filecount manager!